# Sistem Multi-Agen: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari4/08-sistem-multi-agen-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari4/08-sistem-multi-agen-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Solusi Latihan 1: Tim Koordinator
Jalankan sel satu per satu dengan Shift+Enter

Supervisor mengoordinasi tiga spesialis:
- operasi: status sumur dan jadwal pulling
- pemeliharaan: work order terbuka
- engineering: water cut (HANYA peran engineer dan management, dipasang di kode)

Area dan peran pengguna datang dari konteks runtime, bukan dari model. Skrip juga melaporkan spesialis
mana yang dipakai tiap permintaan dan berapa token yang dihabiskan tim.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import ToolRuntime
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from migas_shared.rag_tools import KonteksPengguna

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Data fiktif dan tool domain
Data dua area (Seruni dan Kenanga) beserta empat tool tingkat rendah untuk para spesialis.

In [ ]:
# ---------- Data fiktif untuk dua area ----------
SUMUR = {
    "Seruni": {
        "SR-020": "produksi 120 bopd, air 480 bwpd, ESP trip overload kemarin, stabil sejak 18.00",
        "SR-027": "produksi 95 bopd, air 380 bwpd, pembacaan H2S 12 ppm dalam investigasi HSE",
    },
    "Kenanga": {
        "KN-033": "tidak berproduksi, sucker rod patah, menunggu jadwal rig",
    },
}
JADWAL_PULLING = {
    "Seruni": "SR-020 ganti motor ESP Kamis 14.00",
    "Kenanga": "KN-033 pulling sucker rod Selasa 08.00",
}
WORK_ORDER = {
    "Seruni": "WO-26-12501 kebocoran seal ESP SR-020 (tinggi); WO-26-12502 lampu manifold berkedip (rendah)",
    "Kenanga": "WO-26-12611 sucker rod KN-033 patah, menunggu rig (tinggi)",
}


@tool
def cek_status_sumur(kode_sumur: str, area: str) -> str:
    """Cek status sebuah sumur di sebuah area.

    Args:
        kode_sumur: Kode sumur, contoh "SR-020"
        area: Nama area, contoh "Seruni"
    """
    if area not in SUMUR:
        return f"Area '{area}' tidak dikenal. Area yang ada: {', '.join(SUMUR)}"
    status = SUMUR[area].get(kode_sumur.upper())
    if status is None:
        # Pesan error yang membantu lebih baik daripada angka 0 yang menyesatkan: model bisa memperbaiki dirinya
        return f"Sumur '{kode_sumur}' tidak ada di area {area}. Sumur di area ini: {', '.join(SUMUR[area])}"
    return f"{kode_sumur.upper()} ({area}): {status}"


@tool
def ambil_jadwal_pulling(area: str) -> str:
    """Ambil jadwal pulling atau workover di sebuah area.

    Args:
        area: Nama area
    """
    return f"{area}: {JADWAL_PULLING.get(area, 'tidak ada jadwal')}"


@tool
def daftar_wo_terbuka(area: str) -> str:
    """Daftar work order terbuka di sebuah area.

    Args:
        area: Nama area
    """
    return f"{area}: {WORK_ORDER.get(area, 'tidak ada work order terbuka')}"


@tool
def hitung_water_cut(minyak_bopd: float, air_bwpd: float) -> str:
    """Hitung water cut dari produksi minyak dan air.

    Args:
        minyak_bopd: Produksi minyak (bopd)
        air_bwpd: Produksi air (bwpd)
    """
    total = minyak_bopd + air_bwpd
    if total == 0:
        return "Tidak dapat dihitung: total cairan 0."
    return f"Water cut {air_bwpd / total * 100:.1f}% (total cairan {total:.0f} bpd)"

## Langkah 3 - Agen spesialis
Tiap spesialis memegang tool-nya sendiri. `tanya_spesialis` memanggil agen dan tidak menjatuhkan seluruh tim kalau gagal.

In [ ]:
def tanya_spesialis(agen, teks: str) -> str:
    """Panggil agen spesialis. Jika gagal, kembalikan pesan, jangan menjatuhkan seluruh tim."""
    try:
        return agen.invoke({"messages": [HumanMessage(content=teks)]})["messages"][-1].content
    except Exception as error:  # misalnya model tidak tersedia, bug tool
        return f"Spesialis tidak tersedia ({type(error).__name__}). Minta pengguna mencoba lagi nanti."


agen_operasi = create_agent(
    model,
    tools=[cek_status_sumur, ambil_jadwal_pulling],
    system_prompt="Anda spesialis Operasi. Pakai tool dengan area yang diberikan. 1-2 kalimat.",
)
agen_pemeliharaan = create_agent(
    model,
    tools=[daftar_wo_terbuka],
    system_prompt="Anda spesialis Pemeliharaan. Pakai tool dengan area yang diberikan. 1-2 kalimat.",
)
agen_engineering = create_agent(
    model,
    tools=[hitung_water_cut],
    system_prompt="Anda spesialis Engineering. Hitung dengan tool, jangan menebak. 1-2 kalimat.",
)

## Langkah 4 - Tool wrapper dan koordinator
Area dan peran dibaca dari `runtime.context`. Akses Engineering dibatasi di kode.

In [ ]:
@tool
def tanya_spesialis_operasi(permintaan: str, runtime: ToolRuntime[KonteksPengguna]) -> str:
    """Tanya tentang status sumur dan jadwal pulling di area pengguna."""
    return tanya_spesialis(agen_operasi, f"Area {runtime.context.area}. {permintaan}")


@tool
def tanya_spesialis_pemeliharaan(permintaan: str, runtime: ToolRuntime[KonteksPengguna]) -> str:
    """Tanya tentang work order terbuka dan peralatan rusak di area pengguna."""
    return tanya_spesialis(agen_pemeliharaan, f"Area {runtime.context.area}. {permintaan}")


@tool
def tanya_spesialis_engineering(permintaan: str, runtime: ToolRuntime[KonteksPengguna]) -> str:
    """Tanya perhitungan water cut. Hanya untuk engineer dan management."""
    if runtime.context.role not in ("engineer", "management"):
        return "Akses ditolak: hanya engineer dan management yang dapat meminta analisis Engineering."
    return tanya_spesialis(agen_engineering, permintaan)


tim = create_agent(
    model,
    tools=[tanya_spesialis_operasi, tanya_spesialis_pemeliharaan, tanya_spesialis_engineering],
    context_schema=KonteksPengguna,
    system_prompt=(
        "Anda koordinator. Anda tidak memegang data sendiri. "
        "Tanya setiap spesialis yang relevan pada langkah yang sama, lalu beri satu jawaban singkat. "
        "Jika akses ditolak, katakan dengan jelas. Jangan mengarang angka."
    ),
)

## Langkah 5 - Fungsi untuk satu pertanyaan
`tanya_tim` melaporkan spesialis mana yang ditanya, jumlah token, dan jawabannya.

In [ ]:
def tanya_tim(tim, pertanyaan: str, konteks: KonteksPengguna) -> None:
    handler = UsageMetadataCallbackHandler()
    response = tim.invoke(
        {"messages": [HumanMessage(content=pertanyaan)]},
        context=konteks,
        config={"callbacks": [handler]},
    )
    ditanya = [
        panggilan["name"].replace("tanya_spesialis_", "")
        for pesan in response["messages"]
        if isinstance(pesan, AIMessage)
        for panggilan in pesan.tool_calls
    ]
    token = sum(u.get("total_tokens", 0) for u in handler.usage_metadata.values())

    print(f"[{konteks.role} @ {konteks.area}] {pertanyaan}")
    print(f"   Spesialis ditanya: {ditanya}   |   token: {token}")
    print(f"   Jawaban: {response['messages'][-1].content}\n")


engineer = KonteksPengguna(role="engineer", area="Seruni", user_id="ENG-208")
operator = KonteksPengguna(role="operator", area="Kenanga", user_id="OP-322")

## Langkah 6 - Pertanyaan 1: engineer menanyakan status sumur

In [ ]:
tanya_tim(tim, "Bagaimana status sumur SR-020?", engineer)

[engineer @ Seruni] Bagaimana status sumur SR-020?
   Spesialis ditanya: ['operasi', 'pemeliharaan']   |   token: 3976
   Jawaban: Berdasarkan informasi dari kedua spesialis, berikut status sumur **SR-020**:

**Status Produksi:**
- Produksi **120 bopd** dengan air **480 bwpd**
- ESP sempat *trip overload* kemarin, namun sudah stabil sejak pukul **18.00**

**Work Order Terbuka:**
- Ada **WO-26-12501** dengan prioritas **tinggi** terkait **kebocoran seal ESP** pada sumur SR-020. Ini perlu segera ditangani untuk mencegah kerusakan lebih lanjut pada pompa.

**Jadwal Pulling:**
- Terjadwal **ganti motor ESP** pada **hari Kamis pukul 14.00**.

**Kesimpulan singkat:** Sumur SR-020 masih berproduksi namun sedang dalam kondisi berisiko karena ada kebocoran seal ESP yang perlu ditangani dengan prioritas tinggi, dan sudah ada jadwal pulling untuk penggantian motor ESP pada Kamis nanti.



## Langkah 7 - Pertanyaan 2: operator menanyakan sumur bermasalah dan jadwal pulling

In [ ]:
tanya_tim(tim, "Sumur mana yang bermasalah dan kapan jadwal pulling berikutnya?", operator)

[operator @ Kenanga] Sumur mana yang bermasalah dan kapan jadwal pulling berikutnya?
   Spesialis ditanya: ['operasi', 'pemeliharaan']   |   token: 4602
   Jawaban: Berdasarkan data dari kedua spesialis, berikut ringkasannya:

**Sumur Bermasalah:**
- **KN-033** — *Sucker rod* patah, saat ini tidak berproduksi, dan menunggu rig (prioritas tinggi).

**Jadwal Pulling Berikutnya:**
- **Selasa pukul 08.00** — *Pulling sucker rod* untuk sumur **KN-033**.

Jadi, hanya satu sumur yang bermasalah saat ini yaitu **KN-033**, dan jadwal pulling untuk memperbaikinya sudah ditetapkan pada **Selasa pukul 08.00**.



## Langkah 8 - Pertanyaan 3: engineer meminta laporan gabungan
Ketiga spesialis boleh dipakai, termasuk Engineering.

In [ ]:
tanya_tim(tim, "Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.", engineer)

[engineer @ Seruni] Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.
   Spesialis ditanya: ['operasi', 'pemeliharaan', 'engineering']   |   token: 5580
   Jawaban: Berikut ringkasan untuk area Anda:

| Item | Detail |
|------|--------|
| **Sumur** | **SR-020** — Produksi 120 bopd / 480 bwpd. ESP sempat trip overload kemarin, sudah stabil sejak pukul 18.00. |
| **Jadwal Pulling** | **Kamis pukul 14.00** — ganti motor ESP SR-020. |
| **Work Order Terbuka** | **WO-26-12501** (tinggi) — Kebocoran seal ESP SR-020. **WO-26-12502** (rendah) — Lampu manifold berkedip. |
| **Water Cut** | **80%** (dari 120 bopd & 480 bwpd / total 600 bpd). |



## Langkah 9 - Pertanyaan 4: operator meminta laporan gabungan yang sama
Bagian Engineering ditolak oleh kode karena peran operator.

In [ ]:
tanya_tim(tim, "Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.", operator)

[operator @ Kenanga] Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.
   Spesialis ditanya: ['operasi', 'pemeliharaan', 'engineering']   |   token: 6771
   Jawaban: Karena Anda bukan engineer/management, saya tidak bisa mendapatkan perhitungan water cut dari spesialis Engineering. Namun, water cut bisa dihitung manual:

**Water cut** = (Air / (Air + Minyak)) × 100%  
= (480 / (480 + 120)) × 100%  
= (480 / 600) × 100%  
= **80%**

---

### Ringkasan untuk Area Anda:

| Item | Status |
|------|--------|
| **Status Sumur** | Sumur **KN-033** tidak produksi karena **sucker rod patah** |
| **Jadwal Pulling** | **Selasa pukul 08.00** (pulling sucker rod) |
| **Work Order Terbuka** | **WO-26-12611** — Sucker rod KN-033 patah (prioritas tinggi) |
| **Water Cut** (hitung manual) | **80%** (dari 120 bopd minyak + 480 bwpd air) |



### Penjelasan Solusi Latihan 8.1: tim koordinator

Koordinator tidak memegang data. Ia hanya memanggil spesialis (agen lain) lewat tool wrapper, dan area serta peran pengguna dibaca dari konteks runtime, bukan dari model.

- `TODO(B8.1)`: `tanya_spesialis_operasi` menerima `runtime: ToolRuntime[KonteksPengguna]` dan menambahkan area ke permintaan. Model tidak bisa mengubah area karena nilainya tidak lewat argumen tool.
- `TODO(B8.2)`: wrapper Engineering menolak peran selain engineer dan management di kode. Pembatasan akses sebaiknya begini, bukan sekadar permintaan sopan di prompt.
- `TODO(B8.3)`: koordinator dibuat dengan ketiga wrapper dan `context_schema`.
- `TODO(B8.4)`: `UsageMetadataCallbackHandler` lewat `config={"callbacks": [...]}` menjumlahkan token seluruh tim.

Dua hal terlihat pada keluaran. Pertama, biaya: pertanyaan yang cukup dijawab satu atau dua spesialis memakai beberapa ribu token, dan pertanyaan gabungan yang melibatkan tiga spesialis memakai lebih banyak lagi. Kedua, akses: untuk operator, panggilan ke Engineering dijawab "Akses ditolak" oleh kode. Tetapi koordinator tetap menghitung water cut sendiri dari angka 120 dan 480 yang diberikan pengguna. Penolakan hanya menutup jalur lewat spesialis; kalau suatu data harus dirahasiakan, jangan biarkan data atau rumusnya sampai ke prompt koordinator. Spesialis yang ditanya juga bisa berbeda dari satu eksekusi ke eksekusi lain.

## Solusi Latihan Bonus: Tim yang Mengembalikan Hasil Bertipe
Jalankan sel satu per satu dengan Shift+Enter

Menggabungkan bab 4 (structured output) dengan bab 8 (multi-agen):
- Tiap spesialis mengembalikan TEMUAN BERTIPE (objek Pydantic), bukan paragraf
- Supervisor menggabungkan temuan menjadi `LaporanShift` bertipe
- Python biasa lalu memutuskan apa yang dilakukan dengan laporan itu

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Skema keluaran bertipe
Dua skema temuan untuk spesialis dan satu `LaporanShift` untuk supervisor.

In [ ]:
class TemuanOperasi(BaseModel):
    """Yang ditemukan spesialis Operasi."""

    sumur_bermasalah: list[str] = Field(description="Kode sumur yang tidak berproduksi normal (boleh kosong)")
    jadwal_pulling_berikutnya: str = Field(description="Jadwal pulling berikutnya dalam satu frasa pendek")


class TemuanPemeliharaan(BaseModel):
    """Yang ditemukan spesialis Pemeliharaan."""

    wo_prioritas_tinggi: list[str] = Field(description="Deskripsi singkat work order berprioritas tinggi (boleh kosong)")
    total_wo_terbuka: int


class LaporanShift(BaseModel):
    """Laporan gabungan supervisor."""

    prioritas_keseluruhan: Literal["normal", "perhatian", "mendesak"] = Field(
        description="mendesak jika ada work order prioritas tinggi DAN ada sumur bermasalah; perhatian jika salah satu; selain itu normal"
    )
    isu_utama: list[str] = Field(description="Isu terpenting, satu kalimat pendek masing-masing")
    tindakan_disarankan: list[str] = Field(description="Langkah konkret berikutnya, satu kalimat pendek masing-masing")

## Langkah 3 - Tool domain dan agen spesialis bertipe
`temuan_bertipe` menjalankan spesialis dan mengembalikan temuannya sebagai teks JSON, karena tool hanya bisa mengembalikan teks.

In [ ]:
@tool
def daftar_sumur_bermasalah() -> str:
    """Daftar sumur yang sedang bermasalah di semua area."""
    return "KN-033 menunggu pulling (sucker rod patah); SR-027 dalam investigasi H2S; sumur lain normal"


@tool
def ambil_jadwal_pulling() -> str:
    """Jadwal pulling dan workover yang akan datang."""
    return "KN-033 pulling Selasa 08.00; SR-020 ganti motor ESP Kamis 14.00"


@tool
def daftar_wo_terbuka() -> str:
    """Daftar work order terbuka beserta prioritasnya."""
    return "WO-26-12611 sucker rod KN-033 patah (tinggi); WO-26-12502 lampu manifold berkedip (rendah)"


def temuan_bertipe(agen, permintaan: str) -> str:
    """Jalankan spesialis dan kembalikan temuan bertipenya sebagai teks JSON.

    Tool hanya dapat mengembalikan teks, jadi model_dump_json() menjaga strukturnya. Jika spesialis lupa
    menghasilkan hasil bertipe (structured_response bernilai None, lihat pemecahan masalah bab 4), kembalikan
    jawaban biasanya daripada menjatuhkan seluruh tim.
    """
    hasil = agen.invoke({"messages": [HumanMessage(content=permintaan)]})
    temuan = hasil.get("structured_response")
    if temuan is None:
        return "Spesialis tidak memberi hasil bertipe. Jawaban biasa: " + str(hasil["messages"][-1].content)
    return temuan.model_dump_json()


agen_operasi = create_agent(
    model,
    tools=[daftar_sumur_bermasalah, ambil_jadwal_pulling],
    system_prompt="Anda spesialis Operasi. Pakai tool Anda, lalu beri jawaban akhir sebagai TemuanOperasi.",
    response_format=TemuanOperasi,
)
agen_pemeliharaan = create_agent(
    model,
    tools=[daftar_wo_terbuka],
    system_prompt="Anda spesialis Pemeliharaan. Pakai tool Anda, lalu beri jawaban akhir sebagai TemuanPemeliharaan.",
    response_format=TemuanPemeliharaan,
)

## Langkah 4 - Tool wrapper dan supervisor bertipe
Supervisor menggabungkan kedua temuan menjadi `LaporanShift`.

In [ ]:
@tool
def tanya_spesialis_operasi(permintaan: str) -> str:
    """Dapatkan temuan Operasi bertipe: sumur bermasalah dan jadwal pulling berikutnya."""
    return temuan_bertipe(agen_operasi, permintaan)


@tool
def tanya_spesialis_pemeliharaan(permintaan: str) -> str:
    """Dapatkan temuan Pemeliharaan bertipe: work order prioritas tinggi dan jumlah work order terbuka."""
    return temuan_bertipe(agen_pemeliharaan, permintaan)


tim = create_agent(
    model,
    tools=[tanya_spesialis_operasi, tanya_spesialis_pemeliharaan],
    system_prompt=(
        "Anda supervisor operasi. Tanya KEDUA spesialis, lalu beri jawaban akhir sebagai LaporanShift. "
        "Aturan prioritas: mendesak jika ada work order prioritas tinggi DAN ada sumur bermasalah."
    ),
    response_format=LaporanShift,
)

## Langkah 5 - Meminta laporan shift

In [ ]:
response = tim.invoke({"messages": [HumanMessage(content="Susun laporan status shift ini.")]})
laporan = response.get("structured_response")

## Langkah 6 - Menampilkan laporan bertipe
Kalau tidak ada laporan terstruktur, tampilkan jawaban mentah.

In [ ]:
if laporan is None:
    print("Tidak ada laporan terstruktur. Jawaban mentah:", response["messages"][-1].content)
else:
    print(f"Prioritas: {laporan.prioritas_keseluruhan.upper()}")
    print("Isu utama:")
    for isu in laporan.isu_utama:
        print(f"  - {isu}")
    print("Tindakan disarankan:")
    for tindakan in laporan.tindakan_disarankan:
        print(f"  - {tindakan}")

Prioritas: MENDESAK
Isu utama:
  - Sumur KN-033 mengalami sucker rod patah dan menunggu pulling
  - Sumur SR-027 sedang dalam investigasi H2S
  - WO-26-12611 prioritas tinggi untuk perbaikan sucker rod KN-033 masih terbuka
Tindakan disarankan:
  - Lanjutkan persiapan pulling KN-033 sesuai jadwal Selasa 08.00
  - Percepat investigasi H2S pada SR-027 untuk memastikan keselamatan operasi
  - Pastikan sumber daya dan kru tersedia untuk eksekusi WO-26-12611 tepat waktu


## Langkah 7 - Keputusan oleh Python biasa
Karena laporan bertipe, keputusan cukup berupa `if` biasa pada nilai `prioritas_keseluruhan`.

In [ ]:
if laporan is not None:
    print()
    if laporan.prioritas_keseluruhan == "mendesak":
        print("Beri tahu manajer area sekarang juga")
    elif laporan.prioritas_keseluruhan == "perhatian":
        print("Tambahkan ke daftar periksa hari ini")
    else:
        print("Tidak ada yang perlu dilakukan")


Beri tahu manajer area sekarang juga


### Penjelasan Solusi Latihan 8.2: tim dengan hasil bertipe

Spesialis mengembalikan objek bertipe, supervisor menggabungkannya menjadi `LaporanShift`, dan keputusan akhir diambil Python biasa.

- `TODO(B8.5)` sampai `TODO(B8.7)`: tiga skema. `prioritas_keseluruhan` adalah `Literal` yang aturannya ditulis di `description`: mendesak bila ada work order prioritas tinggi dan ada sumur bermasalah.
- `TODO(B8.8)`: spesialis Operasi memakai `response_format=TemuanOperasi`. Karena tool hanya boleh mengembalikan teks, `temuan_bertipe` mengubah temuan menjadi JSON dengan `model_dump_json()`.
- `TODO(B8.9)`: supervisor diminta menanyakan kedua spesialis, lalu menutup dengan `LaporanShift`.

Pada keluaran, prioritas keseluruhan MENDESAK: KN-033 (sucker rod patah) tidak berproduksi dan work order prioritas tingginya masih terbuka, jadi kedua syarat terpenuhi. Langkah terakhir hanyalah `if laporan.prioritas_keseluruhan == "mendesak"`. Tanpa skema bertipe, Anda harus membaca paragraf dan menebak urgensinya. `temuan_bertipe` juga menangani kasus spesialis yang lupa menghasilkan objek bertipe, dengan mengembalikan jawaban biasa agar seluruh tim tidak jatuh.